In [1]:
!pip -q install gradio

import pandas as pd
import numpy as np
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

data = df[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()
data["Age"] = data["Age"].fillna(data["Age"].median())
data["Embarked"] = data["Embarked"].fillna(data["Embarked"].mode()[0])
data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
data["Sex_num"] = data["Sex"].map({"male": 0, "female": 1})
data = pd.get_dummies(data, columns=["Embarked"], drop_first=True)

feature_cols = ["Pclass", "Sex_num", "Age", "SibSp", "Parch", "FamilySize", "Fare"] + [
    c for c in data.columns if c.startswith("Embarked_")
]

X = data[feature_cols]
y = data["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
model.fit(X_train, y_train)

acc = accuracy_score(y_test, model.predict(X_test))
print(f"Model Test Accuracy: {acc * 100:.2f}%")

Model Test Accuracy: 77.65%


In [2]:
def predict_survival(pclass, sex, age, sibsp, parch, fare, embarked):
    family_size = sibsp + parch + 1
    sex_num = 1 if sex == "female" else 0

    row = {
        "Pclass": int(pclass),
        "Sex_num": sex_num,
        "Age": float(age),
        "SibSp": int(sibsp),
        "Parch": int(parch),
        "FamilySize": int(family_size),
        "Fare": float(fare),
    }

    for col in feature_cols:
        if col.startswith("Embarked_"):
            row[col] = 1 if col == f"Embarked_{embarked}" else 0

    input_df = pd.DataFrame([row])[feature_cols]
    pred = int(model.predict(input_df)[0])
    prob = float(model.predict_proba(input_df)[0][1])

    status = "Survived (1)" if pred == 1 else "Did Not Survive (0)"
    return f"Prediction: {status}\nEstimated Survival Probability: {prob * 100:.2f}%"

app = gr.Interface(
    fn=predict_survival,
    inputs=[
        gr.Slider(1, 3, step=1, value=1, label="Passenger Class (1 = 1st, 2 = 2nd, 3 = 3rd)"),
        gr.Dropdown(["female", "male"], value="female", label="Sex"),
        gr.Slider(1, 80, step=1, value=28, label="Age"),
        gr.Slider(0, 8, step=1, value=0, label="Siblings / Spouses Aboard (SibSp)"),
        gr.Slider(0, 6, step=1, value=0, label="Parents / Children Aboard (Parch)"),
        gr.Slider(0, 500, step=5, value=50, label="Ticket Fare"),
        gr.Dropdown(["S", "C", "Q"], value="S", label="Port of Embarkation"),
    ],
    outputs=gr.Textbox(label="Real-Time Prediction Output", lines=3),
    title="Titanic Survival Predictor (ML Web App)",
    description="Adjust passenger features using the sliders and dropdowns to get real-time survival predictions.",
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://afd33f47e36287f162.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
